# 추론 → JSON + 오버레이 이미지 (프론트엔드 납품물)

## 프론트 질문: "ROI와 bbox 좌표계가 동일한가?" → **예, 동일합니다.**

- 우리가 납품하는 **이미지 자체가 이미 ROI**입니다. (CT는 전처리에서 ROI로 crop된 상태로 저장됨)
- `bbox_xyxy` 는 **그 ROI 이미지의 픽셀 좌표** (origin = 좌상단, 단위 = px).
- 즉 `image.width x image.height` = ROI 크기 = bbox가 사는 좌표계. **변환·오프셋 불필요, 그대로 그리면 맞습니다.**
- EXT(RGB)는 crop 자체가 없어 이미지 = ROI = 전체 프레임 (동일).

JSON 매 파일에 `coordinate_space` 블록으로 이 계약을 명시하고, §4에서 **이미지 크기 == manifest의 roi_w/roi_h** 임을 assert로 검증합니다.

§1 셋업 → §2 추론·JSON → §3 오버레이 이미지 → §4 자체검증(좌표 정합성)


In [ ]:
# == §1 셋업: 경로 자동탐색 (가중치·manifest·이미지) ==
# 못 찾으면 실제 폴더 구조를 출력하니, 보고 ★수동지정 3줄만 채우면 됨.
!pip -q install ultralytics sahi pandas
import os, re, json, subprocess, zipfile
from pathlib import Path
import pandas as pd
from google.colab import drive
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')

ROOT      = Path('/content/drive/MyDrive/battery_yolo')
DRIVE_OUT = ROOT/'kt_out_1'
RUNS      = DRIVE_OUT/'runs_main'
assert RUNS.exists(), f'runs_main 없음: {RUNS}'

# ── ★수동지정 (None이면 자동탐색) ──
WEIGHT  = None      # 예: RUNS/'train_ct_tiled_v36_nobig/weights/best.pt'
MF      = None      # 예: ROOT/'data/battery_v41_output/manifest.csv'
SRC     = None      # 이미지가 든 폴더 또는 zip. 예: ROOT/'data/battery_v41_output/CT'

# ── 추론 설정 ──
MODE   = 'sahi'      # 'sahi'=CT | 'plain'=EXT
MODAL  = 'CT'        # 'CT' | 'EXT'
CONF   = 0.05        # ★CT 배포 운영점 = samedist ep6 @ slice1280/ov0.2 (0728 재측정 확정)
                     #   0.05 = 비용비대칭 반영(F2·F3 동시 최적): R 0.977 / loc 86.7% / 재검부하 6.6%
                     #   0.10 = F1 최대 0.921 (모델 비교용, 배포용 아님) | 0.02 = R 0.990, 재검 11.8%
                     #   ⚠️ 이 값은 samedist ep6 + 아래 SLICE/OVERLAP 조합 전용. EXT면 0.25
SLICE, OVERLAP = 1280, 0.2   # ★학습 TILE(1280)과 일치. 1024/0.4는 recall↓·오탐 2배(0728 §2)
IMGSZ  = 1280        # plain(EXT)
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

# ══ 1) 가중치 ═══════════════════════════════════════════════════════════
if WEIGHT is None:
    MYDRIVE = Path('/content/drive/MyDrive')
    # ★0728: CONF 0.10이 검증된 모델 = samedist ep6. 이게 있으면 최우선.
    PRI = ([MYDRIVE/'kt_out/champions/ct_samedist_CHAMPION_ep6.pt',
            ROOT/'champions/ct_samedist_CHAMPION_ep6.pt',
            DRIVE_OUT/'champions/ct_samedist_CHAMPION_ep6.pt',
            RUNS/'train_ct_tiled_v36_nobig/weights/best.pt',
            DRIVE_OUT/'ct_tiled_v36_nobig_best_backup.pt'] if MODAL == 'CT' else [])
    for c in PRI:
        if c.exists(): WEIGHT = c; break
    if WEIGHT is None:
        pats = (['*v36*nobig*/weights/best.pt', '*nobig*/weights/best.pt']
                if MODAL == 'CT' else ['*ext*/weights/best.pt', '*ext*best*.pt'])
        for pat in pats:
            cs = [p for p in sorted(RUNS.glob(pat)) if 'p2' not in p.parent.parent.name.lower()]
            if cs: WEIGHT = cs[-1]; break
assert WEIGHT and Path(WEIGHT).exists(), '★가중치 못찾음 — WEIGHT 직접 지정'
WEIGHT = Path(WEIGHT)
print('✅ WEIGHT:', WEIGHT)
# ── 운영점 ↔ 가중치 짝 확인 (CONF는 모델별로 다름 — 잘못 짝지으면 조용히 틀린 리포트가 나옴) ──
if MODAL == 'CT':
    _sd = 'samedist' in str(WEIGHT).lower() or 'CHAMPION_ep6' in str(WEIGHT)
    if _sd:
        print(f'   운영점 짝 OK: samedist ep6 + CONF {CONF} @ slice{SLICE}/ov{OVERLAP}'
              f'  (F2·F3 최적 · R 0.977 · loc 86.7% · 재검부하 6.6%)')
    else:
        print(f'   ⚠️ 경고: CONF {CONF}는 samedist ep6 전용 운영점인데 다른 가중치가 선택됨.')
        print(f'      → samedist 챔피언을 백업했는지 확인하거나, 이 모델의 conf 스윕으로 운영점을 다시 잡을 것.')

# ══ 2) 데이터 폴더 탐색 + 실제 구조 출력 ════════════════════════════════
DATA = None
for c in sorted((ROOT/'data').glob('*')) if (ROOT/'data').exists() else []:
    if c.is_dir() and 'v4' in c.name: DATA = c; break
if DATA is None and (ROOT/'data').exists():
    d = [c for c in sorted((ROOT/'data').glob('*')) if c.is_dir()]
    DATA = d[0] if d else None
assert DATA, f'★data 폴더 없음: {ROOT}/data'
print('✅ DATA:', DATA)

print('\n■ DATA 구조 (2단계):')
for p in sorted(DATA.glob('*'))[:20]:
    kind = 'DIR ' if p.is_dir() else f'{p.stat().st_size/1e6:6.1f}MB'
    print(f'   {kind} {p.name}')
    if p.is_dir():
        for q in sorted(p.glob('*'))[:6]:
            print(f'          └ {q.name}{"/" if q.is_dir() else ""}')

# ══ 3) manifest (선택 — 없어도 추론·JSON은 정상) ═════════════════════════
def find_manifest():
    for base in [DATA, ROOT/'data', ROOT]:
        if not base.exists(): continue
        h = sorted(base.rglob('manifest.csv'))
        if h: return h[0]
    for base in [DATA, ROOT/'data']:
        if not base.exists(): continue
        for z in sorted(base.rglob('*.zip')):
            try: zf = zipfile.ZipFile(z)
            except zipfile.BadZipFile: continue
            for n in zf.namelist():
                if n.replace('\\', '/').split('/')[-1] == 'manifest.csv':
                    zf.extract(n, '/content/work/meta'); return Path('/content/work/meta')/n
    return None
if MF is None: MF = find_manifest()
MANI = {}
if MF and Path(MF).exists():
    _m = pd.read_csv(MF, dtype=str, keep_default_na=False)
    MANI = {r['output_image_name']: r for _, r in _m.iterrows()}
    print(f'\n✅ manifest: {MF} ({len(MANI)}행)')
else:
    print('\n⚠️ manifest 없음 → §4의 ROI 크기 대조만 skip (JSON·이미지는 정상 생성)')

# ── 셀별 3D 부피 유도 (축 3개의 이미지 크기를 교차 대조) ──────────────────
#  x스캔 이미지 = (Y,Z) / y스캔 = (X,Z) / z스캔 = (X,Y)  → 세 축이 서로를 검증
#  ⚠️ 셀마다 ROI 크기가 다름(101=562, 중앙값=411) → 하드코딩 금지, 셀별 계산
VOL = {}
if MANI:
    _c = _m[_m['modality'] == 'CT'].copy()
    _c['idx'] = _c['original_stem'].str.extract(r'_[xyz]_(\d+)$')[0].astype(float)
    for _c_ in ['roi_w', 'roi_h']:
        _c[_c_] = pd.to_numeric(_c[_c_], errors='coerce')
    for cid, g in _c.groupby('battery_id'):
        ax = {a: dict(w=int(gg['roi_w'].median()), h=int(gg['roi_h'].median()),
                      n=int(gg['idx'].max()) + 1)
              for a, gg in g.groupby('axis') if gg['roi_w'].notna().any()}
        X = ax.get('y', {}).get('w') or ax.get('z', {}).get('w')     # y·z 스캔의 폭
        Y = ax.get('x', {}).get('w') or ax.get('z', {}).get('h')     # x 스캔의 폭 = z 스캔의 높이
        Z = ax.get('x', {}).get('h') or ax.get('y', {}).get('h')     # x·y 스캔의 높이
        if not (X and Y and Z): continue
        VOL[cid] = {'dims_px': {'X': X, 'Y': Y, 'Z': Z},
                    'slices': {a: ax[a]['n'] for a in ax}}
    if VOL:
        _k = next(iter(VOL)); _v = VOL[_k]
        print(f"✅ 3D 부피 유도: {len(VOL)}개 셀 | 예(셀 {_k}) "
              f"X{_v['dims_px']['X']}×Y{_v['dims_px']['Y']}×Z{_v['dims_px']['Z']}, 슬라이스 {_v['slices']}")

# ══ 4) 이미지 소스 ══════════════════════════════════════════════════════
# 우선순위: ①SRC 지정 ②이미 해제된 로컬(/content/work/data/ct = samedist 노트북 산출)
#           ③Drive에 흩어진 loose 이미지 ④zip 해제
IMG_DIR = None; IMGS = []
def scan(d): return sorted(f for f in Path(d).rglob('*') if f.suffix.lower() in IMG_EXTS)

cands = []
if SRC: cands.append(Path(SRC))
cands += [Path('/content/work/data/ct'), Path('/content/work/infer_src')]   # 같은 세션에서 §0 돌렸으면 여기 있음
for c in cands:
    if c.exists() and c.is_dir():
        f = scan(c)
        if f: IMG_DIR, IMGS = c, f; print(f'\n✅ 이미지 소스(기존): {c} — {len(f)}장'); break

if not IMGS:                      # Drive에 loose 이미지가 있나
    f = scan(DATA)
    if f: IMG_DIR, IMGS = DATA, f; print(f'\n✅ 이미지 소스(Drive loose): {DATA} — {len(f)}장')

if not IMGS:                      # zip 해제
    zips = [Path(SRC)] if (SRC and str(SRC).endswith('.zip')) else []
    if not zips:
        key = 'CT' if MODAL == 'CT' else 'RGB'
        pool = sorted(DATA.rglob('*.zip'))
        print('\n■ zip 후보:', [z.name for z in pool[:15]] or '(없음)')
        zips = ([z for z in pool if key.lower() in z.name.lower() and 'test' in z.name.lower()]
                or [z for z in pool if key.lower() in z.name.lower()] or pool)
    IMG_DIR = Path('/content/work/infer_src'); IMG_DIR.mkdir(parents=True, exist_ok=True)
    for z in zips[:1]:
        print('해제:', z)
        subprocess.run(['unzip', '-q', '-o', str(z), '-d', str(IMG_DIR)])
        IMGS = scan(IMG_DIR)
        if IMGS: break

assert IMGS, ('★이미지 못찾음. 위 "DATA 구조"를 보고 SRC= 로 이미지 폴더(또는 zip)를 직접 지정하세요.\n'
              f'   찾아본 곳: {[str(c) for c in cands]} / {DATA}')

# ── 납품 샘플: x·y·z 3축 골고루 (프론트가 3축 케이스를 다 봐야 함) ──
# 실측(v4.1 test 7셀): x 1,261장 결함 0 / y 5,030장 결함 1,287 / z 4,942장 결함 920
#   → x축은 결함 라벨이 없으므로 PASS 예시로만 포함.
N_PER_AXIS = {'y': 8, 'z': 8, 'x': 4}      # 합 20장. 늘리려면 여기만
AXIS_RE = re.compile(r'_([xyz])_(\d+)')

def axis_of(p):
    m = AXIS_RE.search(p.name)
    return m.group(1) if m else '?'
def is_defect(p):
    return str(MANI.get(p.name, {}).get('has_porosity', '')).lower() == 'true' if MANI else False

pool = {}
for p in IMGS:
    pool.setdefault(axis_of(p), []).append(p)

picked = []
for ax, n in N_PER_AXIS.items():
    cand = pool.get(ax, [])
    if not cand: continue
    d = [p for p in cand if is_defect(p)]           # 결함 우선
    c = [p for p in cand if not is_defect(p)]
    picked += (d + c)[:n]
IMGS = picked or IMGS[:sum(N_PER_AXIS.values())]

from collections import Counter
print(f'\n✅ 추론 대상 {len(IMGS)}장 | 축 분포:', dict(Counter(axis_of(p) for p in IMGS)))
if MANI:
    print(f'   결함 있는 이미지 {sum(is_defect(p) for p in IMGS)}장 / PASS {sum(not is_defect(p) for p in IMGS)}장')
print(f'   예: {IMGS[0].name}')

# ── 출력 폴더: kt_out_1은 공유 shortcut이라 읽기전용일 수 있음 → 쓰기 가능한 곳 자동 선택 ──
def pick_writable(name):
    for c in [DRIVE_OUT/name, ROOT/name,
              Path('/content/drive/MyDrive')/f'kt_out_mine/{name}',
              Path('/content')/name]:
        try:
            c.mkdir(parents=True, exist_ok=True)
            t = c/'.wtest'; t.write_text('ok'); t.unlink()
            return c
        except Exception:
            continue
    raise RuntimeError('쓰기 가능한 출력 폴더를 못 찾음')

OUTDIR = pick_writable('infer_json')
(OUTDIR/'json').mkdir(parents=True, exist_ok=True)
print('✅ 출력:', OUTDIR)
if str(OUTDIR).startswith('/content/') and 'drive' not in str(OUTDIR):
    print('   ⚠️ 로컬 디스크입니다(세션 종료 시 삭제). §3 끝에서 zip 다운로드하세요.')


In [ ]:
!pip -q install sahi ultralytics
# == §2 추론 → JSON (coordinate_space 계약 명시) ==
from PIL import Image
# ── 선행 셀 확인 (없으면 여기서 명확히 실패) ──
_need = [n for n in ('CONF', 'IMGS', 'IMGSZ', 'MODAL', 'MODE', 'OUTDIR', 'OVERLAP', 'SLICE') if n not in globals()]
assert not _need, f'★앞 셀을 먼저 실행하세요 — 없는 변수: {_need}'

Image.MAX_IMAGE_PIXELS = None
SCHEMA_VERSION = '1.0'
# CT 파일명 = CT_cell_pouch_<셀ID>_<축>_<슬라이스번호>
#   슬라이스 번호가 있어야 프론트가 3D 위치를 특정할 수 있음(2D bbox만으론 '몇 번째 단면'을 모름).
CT_RE  = re.compile(r'CT_cell_[^_]+_(\d+)_([xyz])_(\d+)')
EXT_RE = re.compile(r'RGB_cell_[^_]+_(\d+)_(\d+)')

def parse_name(name):
    m = CT_RE.search(name)
    if m: return m.group(1), m.group(2), int(m.group(3))
    m = EXT_RE.search(name)
    if m: return m.group(1), None, int(m.group(2))
    return None, None, None

def coord_block(W, H):
    """프론트가 읽을 좌표 계약. 납품 이미지 = ROI, bbox = 그 ROI의 픽셀 좌표."""
    return {
        'origin': 'top-left',
        'unit': 'pixel',
        'reference': 'roi_image',
        'bbox_format': 'xyxy',
        'roi_size': [W, H],
        'note': (f'납품 이미지 자체가 ROI({W}x{H})이며, bbox_xyxy/segmentation 좌표는 '
                 '이 ROI 이미지의 픽셀 좌표입니다. 좌표 변환·오프셋 없이 그대로 사용하세요.'),
    }

# 축별 3D 매핑: 단면 이미지는 2개 축만 보여주고, 나머지 1개를 slice_index가 채운다
AXIS_MAP = {'x': {'bbox_x': 'Y', 'bbox_y': 'Z', 'slice_index': 'X'},
            'y': {'bbox_x': 'X', 'bbox_y': 'Z', 'slice_index': 'Y'},
            'z': {'bbox_x': 'X', 'bbox_y': 'Y', 'slice_index': 'Z'}}

def volume_block(cid, axis):
    """프론트가 JSON만 보고 3D 좌표를 계산할 수 있게 하는 블록."""
    v = VOL.get(str(cid)) if 'VOL' in globals() else None
    if v is None or axis not in AXIS_MAP: return None
    mp = AXIS_MAP[axis]
    depth_axis = mp['slice_index']                    # slice_index가 담당하는 축
    n = v['slices'].get(axis)
    scale = round(v['dims_px'][depth_axis] / n, 4) if n else None
    return {
        'dims_px': v['dims_px'],
        'axis_mapping': mp,
        'slice_axis': depth_axis,
        'slice_count': n,
        'slice_scale_px': scale,
        'note': (f"3D 좌표 = bbox_x→{mp['bbox_x']}, bbox_y→{mp['bbox_y']}, "
                 f"slice_index×{scale}→{depth_axis} (픽셀/복셀 단위). "
                 "물리 단위(mm)는 스캔 메타데이터(voxel spacing)가 원본에 없어 제공 불가. "
                 "slice_scale_px는 슬라이스가 ROI를 균등히 덮는다는 가정의 근사값."),
    }

def dets_sahi(path):
    from sahi import AutoDetectionModel
    from sahi.predict import get_sliced_prediction
    global _M
    if '_M' not in globals():
        _M = AutoDetectionModel.from_pretrained(model_type='ultralytics',
                model_path=str(WEIGHT), confidence_threshold=CONF, device='cuda:0')
    r = get_sliced_prediction(str(path), _M, slice_height=SLICE, slice_width=SLICE,
            overlap_height_ratio=OVERLAP, overlap_width_ratio=OVERLAP,
            postprocess_match_threshold=0.5, verbose=0)
    out = []
    for op in r.object_prediction_list:
        seg = None
        try:
            if getattr(op, 'mask', None) is not None: seg = op.mask.segmentation
        except Exception: pass
        out.append((op.category.name, float(op.score.value), op.bbox.to_xyxy(), seg))
    return out

def dets_plain(path):
    from ultralytics import YOLO
    global _Y
    if '_Y' not in globals(): _Y = YOLO(str(WEIGHT))
    r = _Y.predict(str(path), imgsz=IMGSZ, conf=CONF, verbose=False)[0]
    out = []
    if r.boxes is not None:
        xy = r.boxes.xyxy.cpu().numpy(); cf = r.boxes.conf.cpu().numpy(); cl = r.boxes.cls.cpu().numpy()
        polys = [p.reshape(-1).tolist() for p in r.masks.xy] if getattr(r, 'masks', None) is not None else None
        for i in range(len(xy)):
            out.append((r.names[int(cl[i])], float(cf[i]), [float(v) for v in xy[i]],
                        ([polys[i]] if polys else None)))
    return out

run = dets_sahi if MODE == 'sahi' else dets_plain
# 챔피언 백업은 runs 폴더 밖에 있어 경로에서 run 이름이 안 나옴 → 옆의 메타 사이드카에서 읽는다
_side = WEIGHT.with_suffix('.json')
_run = WEIGHT.parent.parent.name
if _side.exists():
    try: _run = json.loads(_side.read_text(encoding='utf-8')).get('run', _run)
    except Exception: pass
model_meta = {
    'weight': WEIGHT.name, 'run': _run,
    'inference': (f'SAHI sliced (slice={SLICE}, overlap={OVERLAP})' if MODE == 'sahi'
                  else f'plain (imgsz={IMGSZ})'),
    'confidence_threshold': CONF,
}

all_json = []
for k, ip in enumerate(IMGS):
    W, H = Image.open(ip).size
    cid, axis, sidx = parse_name(ip.name)
    ds = []
    for i, (cls, conf, (x1, y1, x2, y2), seg) in enumerate(run(ip)):
        w, h = x2 - x1, y2 - y1
        d = {
            'id': i, 'class': cls, 'confidence': round(conf, 4),
            'bbox_xyxy': [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)],
            'bbox_xywh': [round(x1, 1), round(y1, 1), round(w, 1), round(h, 1)],
            'bbox_norm_cxcywh': [round((x1 + w / 2) / W, 6), round((y1 + h / 2) / H, 6),
                                 round(w / W, 6), round(h / H, 6)],
            'area_px': round(w * h, 1),
        }
        if seg: d['segmentation'] = [[round(v, 1) for v in poly] for poly in seg]
        ds.append(d)
    rec = {
        'schema_version': SCHEMA_VERSION,
        'image': {'file': ip.name, 'width': W, 'height': H},
        'coordinate_space': coord_block(W, H),
        'cell': {'id': cid, 'modality': MODAL, 'axis': axis, 'slice_index': sidx,
                 'slice_note': ('3D 위치 = axis + slice_index + bbox(2D). '
                                'CT 한 장은 단면이라 bbox는 2축뿐이고, 나머지 1축을 slice_index가 채움.')},
        'volume': volume_block(cid, axis),
        'model': model_meta,
        'verdict': 'REJECT' if ds else 'PASS',
        'num_detections': len(ds),
        'detections': ds,
    }
    (OUTDIR / 'json' / f'{ip.stem}.json').write_text(
        json.dumps(rec, ensure_ascii=False, indent=2), encoding='utf-8')
    all_json.append(rec)
    if (k + 1) % 20 == 0: print(f'  {k+1}/{len(IMGS)}')

(OUTDIR / 'detections_all.json').write_text(
    json.dumps(all_json, ensure_ascii=False, indent=2), encoding='utf-8')
nrej = sum(r['verdict'] == 'REJECT' for r in all_json)
print(f"\n[OK] {len(all_json)}장 → {OUTDIR}/json/*.json + detections_all.json")
print(f"REJECT {nrej} / PASS {len(all_json)-nrej} | 총 검출 {sum(r['num_detections'] for r in all_json)}건")
ex = next((r for r in all_json if r['detections']), all_json[0])
print('\n───── 실제 출력 예시 (검출 1건까지만) ─────')
print(json.dumps({**ex, 'detections': ex['detections'][:1]}, ensure_ascii=False, indent=2)[:1800])


In [ ]:
# == §3 오버레이 이미지 (JSON의 bbox_xyxy만 사용 = 좌표 계약 증명) ==
from PIL import Image, ImageDraw, ImageFont
# ── 선행 셀 확인 (없으면 여기서 명확히 실패) ──
_need = [n for n in ('IMG_DIR', 'IMG_EXTS', 'OUTDIR') if n not in globals()]
assert not _need, f'★앞 셀을 먼저 실행하세요 — 없는 변수: {_need}'

VIZ = OUTDIR / 'viz'; VIZ.mkdir(parents=True, exist_ok=True)
ONLY_DEFECT = True     # 검출 있는 것만 저장(리포트용)
DRAW_SEG = True
COLORS = {'porosity': (255, 60, 60), 'Damaged': (255, 60, 60), 'Pollution': (60, 140, 255)}

recs = json.loads((OUTDIR / 'detections_all.json').read_text(encoding='utf-8'))
srcmap = {f.stem: f for f in IMG_DIR.rglob('*') if f.suffix.lower() in IMG_EXTS}

def font(sz):
    for p in ('/root/.config/Ultralytics/Arial.ttf',
              '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'):
        try: return ImageFont.truetype(p, sz)
        except Exception: pass
    return ImageFont.load_default()

n = 0
for rec in recs:
    if ONLY_DEFECT and not rec['detections']: continue
    sp = srcmap.get(Path(rec['image']['file']).stem)
    if sp is None: continue
    im = Image.open(sp).convert('RGB'); W, H = im.size
    assert (W, H) == (rec['image']['width'], rec['image']['height']), \
        f"이미지 크기 불일치 {sp.name}: 실제 {(W,H)} vs JSON {(rec['image']['width'], rec['image']['height'])}"
    dr = ImageDraw.Draw(im)
    lw = max(2, round(max(W, H) / 500)); fs = max(14, round(max(W, H) / 80)); ft = font(fs)
    for d in rec['detections']:
        col = COLORS.get(d['class'], (255, 200, 0))
        x1, y1, x2, y2 = d['bbox_xyxy']          # ★ JSON 값 그대로 사용(변환 없음)
        dr.rectangle([x1, y1, x2, y2], outline=col, width=lw)
        if DRAW_SEG and d.get('segmentation'):
            for poly in d['segmentation']:
                if len(poly) >= 6: dr.line(list(poly) + poly[:2], fill=col, width=max(1, lw // 2))
        tag = f"{d['class']} {d['confidence']:.2f}"
        ty = max(0, y1 - fs - 2)
        dr.rectangle([x1, ty, x1 + len(tag) * fs * 0.6, ty + fs + 2], fill=col)
        dr.text((x1 + 1, ty), tag, fill=(255, 255, 255), font=ft)
    im.save(VIZ / f"{Path(rec['image']['file']).stem}.jpg", quality=90)
    n += 1
print(f'[OK] 오버레이 {n}장 → {VIZ}')
print('JSON의 bbox_xyxy를 좌표 변환 없이 그대로 그림 → 박스가 결함에 맞으면 ROI-bbox 좌표계 일치 확인.')

# 노트북에 바로 표시 (납품물 눈으로 확인)
from IPython.display import display
for s in sorted(VIZ.glob('*.jpg'))[:2]:
    full = Image.open(s); print(s.name, full.size)
    th = full.copy(); th.thumbnail((700, 700)); display(th)


# ── 납품 번들: json + viz 를 zip 하나로 (로컬 출력이면 반드시 내려받을 것) ──
import shutil
bundle = shutil.make_archive(str(OUTDIR/'infer_bundle'), 'zip', str(OUTDIR))
print('번들:', bundle, f'({Path(bundle).stat().st_size/1e6:.1f} MB)')
if str(OUTDIR).startswith('/content/') and 'drive' not in str(OUTDIR):
    from google.colab import files; files.download(bundle)


In [ ]:
# == §4 자체검증: ROI == bbox 좌표계 (프론트 납품 전 게이트) ==
import json
# ── 선행 셀 확인 (없으면 여기서 명확히 실패) ──
_need = [n for n in ('MANI', 'OUTDIR') if n not in globals()]
assert not _need, f'★앞 셀을 먼저 실행하세요 — 없는 변수: {_need}'

recs = json.loads((OUTDIR / 'detections_all.json').read_text(encoding='utf-8'))
bad_box, bad_roi = [], []
n_det = 0
for r in recs:
    W, H = r['image']['width'], r['image']['height']
    cs = r['coordinate_space']
    assert cs['origin'] == 'top-left' and cs['bbox_format'] == 'xyxy'
    assert cs['roi_size'] == [W, H], f"roi_size != 이미지 크기: {r['image']['file']}"

    # (1) 납품 이미지가 정말 ROI인가 — manifest의 roi_w/roi_h와 대조
    row = MANI.get(r['image']['file'])
    if row is not None and (int(row['roi_w']), int(row['roi_h'])) != (W, H):
        bad_roi.append((r['image']['file'], (W, H), (int(row['roi_w']), int(row['roi_h']))))

    # (2) bbox가 그 ROI 좌표계 안에 있는가 + 정규화값과 픽셀값이 일치하는가
    for d in r['detections']:
        n_det += 1
        x1, y1, x2, y2 = d['bbox_xyxy']
        if not (0 <= x1 < x2 <= W and 0 <= y1 < y2 <= H):
            bad_box.append((r['image']['file'], d['id'], d['bbox_xyxy'], (W, H)))
        cx, cy, nw, nh = d['bbox_norm_cxcywh']
        assert abs(cx * W - (x1 + x2) / 2) < 1.0 and abs(nw * W - (x2 - x1)) < 1.0, \
            f"정규화·픽셀 좌표 불일치: {r['image']['file']} det{d['id']}"

print(f"검사 {len(recs)}장 / 검출 {n_det}건")
print(f"[1] 이미지 크기 != manifest ROI : {len(bad_roi)}건" if MANI else "[1] manifest 없어 skip")
for b in bad_roi[:5]: print('    ', b)
print(f"[2] ROI 범위 벗어난 bbox        : {len(bad_box)}건")
for b in bad_box[:5]: print('    ', b)
assert not bad_roi, '★ 납품 이미지가 ROI 크기와 다름 — 전처리/소스 확인'
assert not bad_box, '★ bbox가 ROI 범위를 벗어남 — 좌표계 문제'

# (3) 3D 위치 필드가 다 있는가 — 프론트가 단면을 특정하려면 axis+slice_index 필수
miss = [r['image']['file'] for r in recs
        if r['cell'].get('axis') is None or r['cell'].get('slice_index') is None]
print(f"[3] axis/slice_index 누락        : {len(miss)}건")
for x in miss[:5]: print('    ', x)
assert not miss, '★ axis 또는 slice_index 누락 — 파일명 규칙 확인'
from collections import Counter
print('    축 분포:', dict(Counter(r['cell']['axis'] for r in recs)),
      '| slice_index 범위:', min(r['cell']['slice_index'] for r in recs),
      '~', max(r['cell']['slice_index'] for r in recs))

# (4) 3D 매핑 블록 — 프론트가 JSON만으로 3D 좌표를 계산할 수 있는가
vok = [r for r in recs if r.get('volume')]
print(f"[4] volume(3D 매핑) 블록         : {len(vok)}/{len(recs)}건")
if vok:
    v = vok[0]
    d, mp = v['volume']['dims_px'], v['volume']['axis_mapping']
    print(f"    셀 {v['cell']['id']} 부피 X{d['X']}×Y{d['Y']}×Z{d['Z']} | axis={v['cell']['axis']} "
          f"→ bbox_x={mp['bbox_x']}, bbox_y={mp['bbox_y']}, slice={mp['slice_index']} "
          f"(×{v['volume']['slice_scale_px']}px)")
    b = v['detections'][0]['bbox_xyxy'] if v['detections'] else None
    if b:
        sc = v['volume']['slice_scale_px']; si = v['cell']['slice_index']
        print(f"    예시 3D 환산: {mp['bbox_x']}={b[0]:.0f}~{b[2]:.0f}, "
              f"{mp['bbox_y']}={b[1]:.0f}~{b[3]:.0f}, {mp['slice_index']}≈{si*sc:.0f}")
else:
    print("    (manifest 없어 volume 생략 — 3D 좌표 계산 불가, MF 경로 확인)")

s = recs[0]['coordinate_space']
print(f"\n납품 이미지 = ROI {s['roi_size'][0]}x{s['roi_size'][1]}, origin={s['origin']}, "
      f"bbox={s['bbox_format']}, unit={s['unit']}")
print("✅ ROI와 bbox는 동일 좌표계 — 프론트는 좌표 변환 없이 그대로 사용")
